In [13]:
import os
import re
import shutil

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 220)

# ---- المسارات (نصوص) ----
USDA_DIR = "../data/raw/FoodData_Central_sr_legacy_food_csv_2018-04"
MAP_CSV = "ingredient_usda_map.csv"
OVERRIDES_CSV = "manual_overrides.csv"
BACKUP_CSV = "manual_overrides_backup.csv"
NUTRIENTS_PKL = "recipes_nutrients.pkl"
LABELED_PKL = "recipes_labeled.pkl"
BASELINE_CSV = "fixes_baseline.csv"
LOG_CSV = "fixes_log.csv"

for p in [USDA_DIR, MAP_CSV]:
    assert isinstance(p, str), f"{p} must be a string (path string)"
    assert os.path.exists(p), f"Could not find {p}"

In [14]:
food = pd.read_csv(os.path.join(USDA_DIR, "food.csv"))
if "data_type" in food.columns:
    food = food[food["data_type"] == "sr_legacy_food"]
food = food[["fdc_id", "description"]].drop_duplicates("fdc_id").reset_index(drop=True)
desc_by_id = food.set_index("fdc_id")["description"]
id_by_desc = food.drop_duplicates("description").set_index("description")["fdc_id"]

imap = pd.read_csv(MAP_CSV)
known_ingredients = set(imap["ingredient"])

if os.path.exists(OVERRIDES_CSV):
    overrides = pd.read_csv(OVERRIDES_CSV)
else:
    overrides = pd.DataFrame(columns=["ingredient", "fdc_id", "note"])

print(f"USDA foods: {len(food):,} | ingredients in map: {len(imap)} | existing overrides: {len(overrides)}")

USDA foods: 7,793 | ingredients in map: 500 | existing overrides: 1


In [15]:
def snapshot():
    rows = {}
    if os.path.exists(NUTRIENTS_PKL):
        n = pd.read_pickle(NUTRIENTS_PKL)
        rows["recipes_with_nutrients"] = len(n)
        if "quality" in n.columns:
            for q, v in n["quality"].value_counts().items():
                rows[f"quality_{q}"] = int(v)
    if os.path.exists(LABELED_PKL):
        lab = pd.read_pickle(LABELED_PKL)
        rows["labeled_file_rows"] = len(lab)
        for c in lab.columns:
            if lab[c].dtype == bool:
                rows[f"label_{c}"] = int(lab[c].sum())
    return pd.Series(rows, name="value")


if os.path.exists(BASELINE_CSV):
    print(f"{BASELINE_CSV} already exists, not overwritten.")
    baseline = pd.read_csv(BASELINE_CSV, index_col=0)["value"]
else:
    baseline = snapshot()
    baseline.to_csv(BASELINE_CSV, encoding="utf-8-sig")
    print(f"saved {BASELINE_CSV}")
baseline

fixes_baseline.csv موجود من قبل، ما كتبنا فوقه.


recipes_with_nutrients    41911
quality_unverified        17362
quality_poor              10525
quality_good               9508
quality_fair               4516
labeled_file_rows         41911
label_gate_menstrual      25016
label_label_menstrual      1882
label_gate_follicular     22254
label_label_follicular     1830
label_gate_ovulatory      22254
label_label_ovulatory      1830
label_gate_luteal         20631
label_label_luteal         1466
label_implausible          9901
Name: value, dtype: int64

In [22]:
FIXES = [
     # (ingredient, possible alternative names in the mapping file, USDA names in order of preference, fallback search keywords, note)
    ("chicken breast", ["chicken breasts", "boneless skinless chicken breast"],
     ["Chicken, broilers or fryers, breast, meat only, raw"],
     ["chicken", "breast", "meat only", "raw"], "wrong match"),
    ("roasting chicken", ["whole chicken"],
     ["Chicken, roasting, meat and skin, raw", "Chicken, broilers or fryers, meat and skin, raw"],
     ["chicken", "roasting", "raw"], "wrong match"),
    ("whole berry cranberry sauce", ["cranberry sauce"],
     ["Cranberry sauce, canned, sweetened"],
     ["cranberry", "sauce"], "was mapped to worcestershire"),
    ("cool whip", ["frozen whipped topping", "whipped topping"],
     ["Dessert topping, semi solid, frozen"],
     ["topping", "frozen"], "wrong match"),
    ("bisquick", ["biscuit mix", "baking mix"],
     ["Biscuits, plain or buttermilk, dry mix"],
     ["biscuit", "dry mix"], "unmapped"),
    ("brandy", ["cognac"],
     ["Alcoholic beverage, distilled, all (gin, rum, vodka, whiskey) 80 proof"],
     ["alcoholic", "distilled"], "unmapped"),
    ("linguine", ["linguini"],
     ["Pasta, dry, enriched", "Pasta, dry, unenriched"],
     ["pasta", "dry"], "unmapped"),
    ("mangoe", ["mango"],
     ["Mangos, raw"],
     ["mango", "raw"], "synonym: mango"),
    ("oleo", ["oleo margarine"],
     ["Margarine, regular, 80% fat, composite, stick, with salt"],
     ["margarine", "stick"], "synonym: margarine"),
    ("cornflour", ["corn flour"],
     ["Cornstarch"],
     ["cornstarch"], "synonym: cornstarch"),
    ("sultana", ["sultanas", "golden raisin"],
     ["Raisins, seedless", "Raisins, golden, seedless"],
     ["raisins", "seedless"], "synonym: raisin"),
]

MANUAL_PICK = {
    "chicken breast": 171447,
}


def find_ingredient(name, aliases):
    for n in [name] + aliases:
        if n in known_ingredients:
            return n
    return None


def search(words, n=8):
    m = food["description"].str.lower()
    mask = pd.Series(True, index=food.index)
    for w in words:
        mask &= m.str.contains(re.escape(w.lower()))
    return food[mask].assign(n_chars=food.loc[mask, "description"].str.len()).sort_values("n_chars").head(n)


log, need_pick = [], {}
for name, aliases, usda_names, words, note in FIXES:
    ing_name = find_ingredient(name, aliases)
    current = None
    if ing_name is not None:
        cur = imap.loc[imap["ingredient"] == ing_name, "usda_description"]
        current = cur.iloc[0] if len(cur) else None

    fdc, chosen = None, None
    if name in MANUAL_PICK:
        fdc = int(MANUAL_PICK[name])
        chosen = desc_by_id.get(fdc)
    else:
        for d in usda_names:
            if d in id_by_desc.index:
                fdc, chosen = int(id_by_desc[d]), d
                break

    status = ("not in ingredient list" if ing_name is None
              else "fixed" if fdc is not None
              else "needs manual pick")
    if status == "needs manual pick":
        need_pick[name] = words
    log.append({"ingredient": ing_name or name, "status": status, "before": current,
                "after": chosen, "fdc_id": fdc, "note": note})

log = pd.DataFrame(log)
log

,ingredient,status,before,after,fdc_id,note
0,chicken breast,fixed,"Chicken breast, deli, rotisserie seasoned, sliced, prepackaged","Chicken, broilers or fryers, meat and skin, raw",171447,wrong match
1,roasting chicken,fixed,"Chicken, roasting, giblets, raw","Chicken, broilers or fryers, meat and skin, raw",171447,wrong match
2,whole berry cranberry sauce,fixed,"Cranberry sauce, canned, sweetened","Cranberry sauce, canned, sweetened",173961,was mapped to worcestershire
3,cool whip,fixed,"Salad dressing, KRAFT MIRACLE WHIP FREE Nonfat Dressing","Dessert topping, semi solid, frozen",170868,wrong match
4,bisquick,fixed,NaN,"Biscuits, plain or buttermilk, dry mix",174902,unmapped
5,brandy,fixed,NaN,"Alcoholic beverage, distilled, all (gin, rum, vodka, whiskey) 80 proof",174815,unmapped
6,linguine,fixed,NaN,"Pasta, dry, enriched",169736,unmapped
7,mangoe,fixed,NaN,"Mangos, raw",169910,synonym: mango
8,oleo,fixed,NaN,"Margarine, regular, 80% fat, composite, stick, with salt",172346,synonym: margarine
9,cornflour,fixed,NaN,Cornstarch,169698,synonym: cornstarch


In [23]:
for name, words in need_pick.items():
    print(f"\n=== {name} ===")
    print(search(words)[["fdc_id", "description"]].to_string(index=False))
if not need_pick:
     print("All fixes found a USDA name.")

كل الإصلاحات لقت اسم USDA.


In [24]:
# Search ingredient names in the mapping file (for ingredients that weren't found)
def find_in_map(text):
    return imap[imap["ingredient"].str.contains(text, case=False, regex=False)][["rank", "ingredient", "n_recipes", "usda_description"]]

missing = log.loc[log["status"] == "not in ingredient list", "ingredient"].tolist()
for m in missing:
    print(f"\n=== {m} ===")
    key = m.split()[-1]
    res = find_in_map(key)
    print(res.to_string(index=False) if len(res) else "no result")

In [25]:
if os.path.exists(OVERRIDES_CSV) and not os.path.exists(BACKUP_CSV):
    shutil.copy(OVERRIDES_CSV, BACKUP_CSV)
    print(f"backup: {BACKUP_CSV}")

ok = log[log["status"] == "fixed"]
new = pd.DataFrame({"ingredient": ok["ingredient"], "fdc_id": ok["fdc_id"].astype(int),
                    "note": "step3b: " + ok["note"]})

ov = overrides[~overrides["ingredient"].isin(new["ingredient"])]
ov = pd.concat([ov, new], ignore_index=True)
ov["fdc_id"] = ov["fdc_id"].astype(int)
ov.to_csv(OVERRIDES_CSV, index=False, encoding="utf-8-sig")
log.to_csv(LOG_CSV, index=False, encoding="utf-8-sig")

print(f"saved {OVERRIDES_CSV}: {len(ov)} overrides ({len(new)} from this notebook)")
print(log["status"].value_counts().to_dict())

saved manual_overrides.csv: 11 overrides (11 from this notebook)
{'fixed': 11}


In [27]:
after = snapshot()
cmp = pd.DataFrame({"before": baseline, "after": after})
cmp["diff"] = cmp["after"] - cmp["before"]
cmp

,before,after,diff
recipes_with_nutrients,41911,42736,825
quality_unverified,17362,17687,325
quality_poor,10525,10590,65
quality_good,9508,9782,274
quality_fair,4516,4677,161
labeled_file_rows,41911,42736,825
label_gate_menstrual,25016,25556,540
label_label_menstrual,1882,2134,252
label_gate_follicular,22254,22688,434
label_label_follicular,1830,1910,80
